In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

(UserGuide_Tools_Build_SolveAtomsWithAlternateLocations)=
# Solve atoms with alternate locations

*choose coordinates for atoms with alternate locations.*

Structures solved experimentally and deposited in the Protein Data Bank can have atoms with multiple locations. MolSysMT includes a function in the 'build' module to solve this ambiguity: {func}`molsysmt.build.solve_atoms_with_alternate_location`

To illustrate how this function works without a network dependency, we use the bundled 1BNF PDB file:

In [2]:
import molsysmt as msm
from importlib.resources import files
pdb_filename = str(files('molsysmt.data.pdb').joinpath('1bnf.pdb'))

In [3]:
msm.get(pdb_filename, alternate_location=True)

[{'480': {'location_id': array(['A', 'B'], dtype=object),
   'atom_id': ['481', '482'],
   'occupancy': array([0.5, 0.5]),
   'coordinates': <Quantity([[4.17   4.9768 4.186 ]
    [4.1655 4.9841 4.1835]], 'nanometer')>,
   'b_factor': <Quantity([0.2098 0.2083], 'nanometer ** 2')>},
  '481': {'location_id': array(['A', 'B'], dtype=object),
   'atom_id': ['483', '484'],
   'occupancy': array([0.5, 0.5]),
   'coordinates': <Quantity([[4.0741 5.0097 4.2933]
    [4.0777 5.0154 4.2938]], 'nanometer')>,
   'b_factor': <Quantity([0.192  0.1881], 'nanometer ** 2')>},
  '484': {'location_id': array(['A', 'B'], dtype=object),
   'atom_id': ['487', '488'],
   'occupancy': array([0.5, 0.5]),
   'coordinates': <Quantity([[4.1393 5.1182 4.3785]
    [4.1245 5.1383 4.3719]], 'nanometer')>,
   'b_factor': <Quantity([0.1883 0.1695], 'nanometer ** 2')>},
  '485': {'location_id': array(['A', 'B'], dtype=object),
   'atom_id': ['489', '490'],
   'occupancy': array([0.5, 0.5]),
   'coordinates': <Quantity([[4

MolSysMT returns the info about alternate locations as a list of dictionaries where the keys are the indices of the atoms with more than an atom_id, occupancy, b_factor and coordinates (stored in a dictionary in the corresponding values).

The output identifies every canonical atom site with alternate records. Let's load the system as a `molsysmt.MolSys` object to work with that information:

In [4]:
molecular_system = msm.convert(pdb_filename, to_form='molsysmt.MolSys', get_missing_bonds=False)

The resulting object still keeps the information about the alternate locations:

In [5]:
msm.get(molecular_system, element='atom', selection='atom_index==480', alternate_location=True)

[{'480': {'location_id': array(['A', 'B'], dtype=object),
   'atom_id': ['481', '482'],
   'occupancy': array([0.5, 0.5]),
   'coordinates': <Quantity([[4.17   4.9768 4.186 ]
    [4.1655 4.9841 4.1835]], 'nanometer')>,
   'b_factor': <Quantity([0.2098 0.2083], 'nanometer ** 2')>}}]

A PDB conversion keeps the first listed variant as the active coordinates while retaining every variant in `Structures.alternate_location`. Calling the resolver with `location_id='occupancy'` chooses the highest-occupancy variant independently in every selected structure. When occupancies tie at 0.5, location `A` is preferred when available.

In [6]:
msm.get(molecular_system, element='atom', selection=480, coordinates=True)

Magnitude,[[[4.17 4.9768 4.186]]]
Units,nanometer


How can we choose a different location for a specific atom? {func}`molsysmt.build.solve_atoms_with_alternate_location` can help to do it. Let's for instance change all atoms to alternate location "B" to show how this function works:

In [7]:
msm.build.solve_atoms_with_alternate_location(molecular_system, location_id='B')

In [8]:
msm.get(molecular_system, element='atom', selection=480, atom_id=True, coordinates=True)

[['482'], <Quantity([[[4.1655 4.9841 4.1835]]], 'nanometer')>]

The function {func}`molsysmt.build.solve_atoms_with_alternate_location` accepts the input argument 'selection' in case different location ids need to be provided for different atoms:

In [9]:
msm.build.solve_atoms_with_alternate_location(molecular_system, selection=[480,481], location_id=['A','B'])

In [10]:
msm.get(molecular_system, element='atom', selection=[480,481], atom_id=True, coordinates=True)

[['481', '484'],
 <Quantity([[[4.17   4.9768 4.186 ]
   [4.0777 5.0154 4.2938]]], 'nanometer')>]

The input argument 'location_id' accepts an extra value: 'occupancy'. With 'occupancy' each atom takes the location with highest occupancy, or the location id equal to 'A' in case all occupancy values are equal.

In [11]:
msm.build.solve_atoms_with_alternate_location(molecular_system, selection=[480,481], location_id='occupancy')

In [12]:
msm.get(molecular_system, element='atom', selection=[480,481], atom_id=True, coordinates=True)

[['481', '483'],
 <Quantity([[[4.17   4.9768 4.186 ]
   [4.0741 5.0097 4.2933]]], 'nanometer')>]

```{admonition} Warning
:class: danger
Native PDB file, PDB text, and `molsysmt.PDBFileHandler` conversions preserve alternate-location variants. They represent each atom site once in topology and keep the variant atom IDs, occupancies, B factors, and coordinates in the structures layer. Forms that do not declare the `alternate_location` attribute can still lose this metadata; inspect a conversion report when crossing such a boundary.
```

```{admonition} See also
:class: attention
{func}`molsysmt.build.solve_atoms_with_alternate_location`, {func}`molsysmt.basic.get`, {func}`molsysmt.basic.get`
```